# Tarea 14 · Segmentación de clientes con groupby

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/julianschmidt1/tpi-analisis-de-datos/blob/main/notebooks/trabajo/14_segmentacion_groupby.ipynb)

**TPI · Introducción al Análisis de Datos · UTN FRA · División 343**  
**Integrantes:** Matías, Julián y Emmanuel. **Responsable de esta tarea:** Julián.

Comparamos el tamaño, los cargos mensuales y las bajas de los segmentos por tipo de contrato, servicio de internet y método de pago.

## Cómo ejecutarlo

- **Local:** instalá las dependencias de `requirements.txt`, abrí el notebook desde el repo y ejecutá todas las celdas.
- **Colab:** ejecutá todas las celdas. El notebook lee los datos desde GitHub. El botón apunta a `main` y estará disponible cuando se integre la tarea. Para probarla antes, subí el archivo `.ipynb` a Colab.

## 1. Carga del dataset

Usamos `telco_clean.csv` cuando está disponible y `telco_provisorio.csv` durante el trabajo inicial. Leemos las tres dimensiones de segmentación, el cargo mensual y el estado de baja.

In [ ]:
from pathlib import Path
from urllib.error import HTTPError

import pandas as pd

try:
    import google.colab
except ImportError:
    EN_COLAB = False
else:
    EN_COLAB = True

dimensiones = ["Contract", "InternetService", "PaymentMethod"]
variables = dimensiones + ["MonthlyCharges", "Churn"]
ruta_limpia = Path("data/processed/telco_clean.csv")
ruta_provisoria = Path("data/processed/telco_provisorio.csv")

if EN_COLAB:
    url_repo = "https://raw.githubusercontent.com/julianschmidt1/tpi-analisis-de-datos/main/"
    fuente = url_repo + ruta_limpia.as_posix()
    try:
        datos = pd.read_csv(fuente, usecols=variables)
    except HTTPError as error:
        if error.code != 404:
            raise
        fuente = url_repo + ruta_provisoria.as_posix()
        datos = pd.read_csv(fuente, usecols=variables)
else:
    actual = Path.cwd().resolve()
    raiz = next(
        (ruta for ruta in (actual, *actual.parents)
         if (ruta / ruta_limpia).is_file() or (ruta / ruta_provisoria).is_file()),
        None,
    )
    if raiz is None:
        raise FileNotFoundError("Falta el CSV de la tarea 3 o el dataset limpio de la tarea 9.")
    fuente = raiz / ruta_limpia
    if not fuente.is_file():
        fuente = raiz / ruta_provisoria
    datos = pd.read_csv(fuente, usecols=variables)

datos = datos[variables]
print(f"Fuente: {fuente}")

## 2. Agrupación y métricas

`groupby` reúne a los clientes que comparten una categoría. Con `.agg()` contamos clientes, promediamos y sumamos cargos mensuales, y sumamos las bajas. La tasa de baja divide las bajas por la cantidad de clientes de cada grupo. Ordenamos por esa tasa y mostramos el tamaño del grupo para interpretar la comparación.

La suma de cargos mensuales representa los importes mensuales registrados de los clientes del grupo. Incluye a quienes figuran como bajas en el dataset.

In [ ]:
assert datos["Churn"].isin(["Yes", "No"]).all(), "Churn debe contener Yes o No."
datos = datos.assign(baja=datos["Churn"].eq("Yes"))

tablas = {}
for dimension in dimensiones:
    resumen = datos.groupby(dimension, dropna=False, observed=True).agg(
        clientes=("Churn", "size"),
        cargo_mensual_promedio=("MonthlyCharges", "mean"),
        suma_cargos_mensuales=("MonthlyCharges", "sum"),
        bajas=("baja", "sum"),
    )
    resumen["tasa_baja_pct"] = 100 * resumen["bajas"] / resumen["clientes"]
    tablas[dimension] = resumen.sort_values("tasa_baja_pct", ascending=False)

for dimension, resumen in tablas.items():
    print(dimension)
    print(resumen.round(2).to_string())
    print()

## 3. Interpretación con el CSV provisorio

- **Contrato.** El mensual reúne 3.875 clientes, suma 257.294,15 en cargos mensuales registrados y tiene una tasa de baja de 42,71 %. En los contratos de dos años la tasa es de 2,83 %. El tamaño del grupo mensual ayuda a entender por qué concentra 1.655 bajas.
- **Servicio de internet.** Fibra óptica tiene el mayor cargo mensual promedio (91,50) y la mayor tasa de baja (41,89 %), frente a 58,10 y 18,96 % en DSL. Fibra combina cargos mensuales altos y una proporción alta de bajas.
- **Método de pago.** El cheque electrónico reúne 2.365 clientes y 1.071 bajas (45,29 %). Con tarjeta de crédito automática la tasa es de 15,24 %. Para entender la diferencia, se puede comparar a clientes con el mismo tipo de contrato y servicio.

Las tasas son proporciones de bajas registradas dentro de cada grupo. Revisar estas cifras cuando se ejecute con `telco_clean.csv`.